<a href="https://colab.research.google.com/github/Al3xMR/RecuperacionDeInformacion2025B/blob/main/09api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejercicio 9: Uso de la API de Google Gemini

En este ejercicio vamos a aprender a utilizar la API de OpenAI

## 1. Uso básico

El siguiente código sirve para conectarse con la API de Google Gemini de forma básica

In [ ]:
!pip install google-genai

In [2]:
import os
from dotenv import load_dotenv

!pip install python-dotenv -qq

# Load environment variables from the specified path
load_dotenv('/content/gemini.env')

# Access the API key
api_key = os.getenv('GOOGLE_API_KEY')

# Print a message indicating if the key was found (for debugging)
if api_key:
    print("API Key loaded successfully.")
else:
    print("Warning: GOOGLE_API_KEY not found in gemini.env.")

# You can then use this api_key in your genai.Client() call
# For example, in cell e1a6608515f3af8d, you would change:
# client = genai.Client()
# to:
# genai.configure(api_key=api_key)


API Key loaded successfully.


In [3]:
from google import genai

client = genai.Client()
'''
response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="Explain how AI works in a few words",
)

print(response.text)
'''

'\nresponse = client.models.generate_content(\n    model="gemini-3-flash-preview",\n    contents="Explain how AI works in a few words",\n)\n\nprint(response.text)\n'

## 2. Retrieval

### 2.1 Cargo el corpus de 20 News Groups

In [6]:
from sklearn.datasets import fetch_20newsgroups
import pandas as pd

newsgroups = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
df = pd.DataFrame(newsgroups.data, columns=["documento"])
df = df.rename(columns={"documento": "text"})
df.head()

,text
0,\n\nI am sure some bashers of Pens fans are pr...
1,My brother is in the market for a high-perform...
2,\n\n\n\n\tFinally you said what you dream abou...
3,\nThink!\n\nIt's the SCSI card doing the DMA t...
4,1) I have an old Jasmine drive which I cann...


### 2.2 Transformo a embeddings

In [7]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re

# Limpieza básica
def normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_norm"] = df["text"].astype(str).map(normalize_text)

df.head()

,text,text_norm
0,\n\nI am sure some bashers of Pens fans are pr...,I am sure some bashers of Pens fans are pretty...
1,My brother is in the market for a high-perform...,My brother is in the market for a high-perform...
2,\n\n\n\n\tFinally you said what you dream abou...,Finally you said what you dream about. Mediter...
3,\nThink!\n\nIt's the SCSI card doing the DMA t...,Think! It's the SCSI card doing the DMA transf...
4,1) I have an old Jasmine drive which I cann...,1) I have an old Jasmine drive which I cannot ...


In [8]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100):
    """
    Chunking por caracteres.
    max_chars ~ 600-1000 suele funcionar bien.
    overlap ayuda a no cortar ideas a la mitad.
    """
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end]
        chunk = chunk.strip()
        if len(chunk) > 0:
            chunks.append(chunk)
        if end == n:
            break
        start = max(0, end - overlap)
    return chunks

records = []
for i, row in df.iterrows():
    chunks = chunk_text(row["text_norm"], max_chars=800, overlap=100)
    for j, ch in enumerate(chunks):
        records.append({
            "doc_id": int(i),
            "chunk_id": j,
            "text": ch
        })

chunks_df = pd.DataFrame(records)
chunks_df.head(), len(chunks_df)

(   doc_id  chunk_id                                               text
 0       0         0  I am sure some bashers of Pens fans are pretty...
 1       1         0  My brother is in the market for a high-perform...
 2       2         0  Finally you said what you dream about. Mediter...
 3       2         1  urds and Turks once upon a time! Ohhhh so swed...
 4       3         0  Think! It's the SCSI card doing the DMA transf...,
 38871)

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-base-v2"   # recomendado para retrieval
model = SentenceTransformer(MODEL_NAME)

# Textos a indexar (pasajes)
passages = ["passage: " + t for t in chunks_df["text"].tolist()]

In [ ]:
# Embeddings (N x D)
# Se debe usar normalize_embeddings=True para similitud coseno
embeddings = model.encode(
    passages, # quitar el [:1000] para limitar a 1000 entradas
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

In [11]:
print(embeddings.shape, embeddings.dtype)

(38871, 768) float32


In [12]:
def embed_query(query: str) -> np.ndarray:
    q = "query: " + query
    vec = model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    return vec

### 2.3 Creo una query y hago la búsqueda


In [13]:
query_text = "Battery measuring"

query_vec = embed_query(query_text)
query_vec.shape

(1, 768)

In [14]:
# Recuperación con similitud coseno

from sklearn.metrics.pairwise import cosine_similarity

# Similaridad coseno entre query y todos los embeddings
sims = cosine_similarity(query_vec, embeddings)[0]

# Top-k resultados
top_k = 5
top_idx = sims.argsort()[-top_k:][::-1]

# Recuperamos los pasajes más relevantes
retrieved_passages = [chunks_df.iloc[i]["text"] for i in top_idx]
'''
print("Top documentos:")
for i, p in enumerate(retrieved_passages, 1):
    print(f"{i}. {p[:200]}...")
'''

'\nprint("Top documentos:")\nfor i, p in enumerate(retrieved_passages, 1):\n    print(f"{i}. {p[:200]}...")\n'

In [16]:
# Construimos el prompt con query y contexto
prompt = f"""
Query: {query_text}

Context retrieved:
{'\n\n'.join(retrieved_passages)}

Por favor responde usando el contexto anterior.
"""

# Llamada correcta a Gemini
response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=[prompt]   # 👈 lista de strings, no dict
)

print(response.text)


Basado en el contexto proporcionado, el tema de la "medición de baterías" o la evaluación de su rendimiento se aborda desde los siguientes puntos:

1.  **Especificaciones y Objetivos de Salida:**
    Un usuario detalla una meta específica de conversión de energía que requiere medir y alcanzar ciertos valores: partiendo de una batería de coche de **12V @ ~25A**, busca obtener una salida de **250VAC** y, posteriormente, transformarla en voltajes de corriente continua específicos: **+5VDC @ 5A, -5V @ 1A, +12VDC @ 8A y -12VDC @ 1A**.

2.  **Medición de Propiedades Térmicas:**
    El texto menciona que no es solo la temperatura promedio lo que importa, sino la capacidad de transferir calor fuera de la batería. Se sugiere considerar factores medibles como la **conductividad térmica**, la capacidad aislante o la **"masa térmica"** del entorno (comparando, por ejemplo, cómo se siente un suelo de concreto frente a la tierra).

3.  **Validación mediante Pruebas:**
    Se resalta la importancia d

Obtengo los 5 documentos más similares a mi query

In [17]:
print("Top documentos:")
for i, p in enumerate(retrieved_passages, 1):
    print(f"{i}. {p[:200]}...")

Top documentos:
1. bit more of the [mind-boggling] theory? Take care. P.S. My goal is 12V @ ~25A in (car battery) -> 250VAC out and (on the other end) 250V -> +5VDC @ 5A, -5V @ 1A, +12VDC @8A and -12VDC @1A... the dista...
2. I hope David isn't going to be too upset with me for sticking my nose in here again, but here goes......:-) It isn't the average temperature that is the key factor here, but rather which is better at ...
3. : My 9 yr old son has signed up to do a science report on batteries. I was : wondering if anyone could provide me with some information as to how to : construct a home-built battery. In my grade schoo...
4. Yes, 4 points, in really big holes which are fairly clear of most of the other stuff on the board. If you can replace the battery, you can install the battery holder....
5. : My 9 yr old son has signed up to do a science report on batteries. I was : wondering if anyone could provide me with some information as to how to : construct a home-built battery. In m